##### Import Libaries and Functions

In [0]:
# import boto3
import os

from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    row_number,
    when,
    col,
    desc,
    trim,
    min as spark_min,
    max as spark_max,
    first as spark_first,
)
import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

In [0]:
# Import the module
import sys
sys.path.append("../aws_utils")

import aws_utils
# Get Environment Prefix
#
env = aws_utils.get_env_prefix()

##### Create widgets with default values. Change them if needed from UI
* run_date : '2024-04-30'
* unity : True | False

In case of Unity Catalog, the appropriate cluster should be attached

In [0]:
# dbutils.widgets.text("run_date", '2024-04-30')
# dbutils.widgets.text("unity", 'True')

##### Get the values of widgets and initialize the environment variable
Ensure that widgets are assigned the wanted values before proceeding with the next step.

In [0]:
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
#env = os.environ["S3PREFIX"] 

# Takes the values of widgets
run_date = dbutils.widgets.get("run_date")
unity = dbutils.widgets.get("unity")

table_name = 'DUnderLaw'

# '''
# # Initialize the SSM client
# ssm_client = boto3.client("ssm", region_name="eu-central-1")

# response = ssm_client.get_parameter(
#     Name='/cfn/a360/matching/conf_s3_path',
#     WithDecryption=False
# )

# s3_path = response['Parameter']['Value']
# '''



##### Initialize the schema retrieved from Redis

In [0]:
schema=(
     "DUnderLawId string,"
     "SourceSystem string,"
     "PortfolioKey string,"
     "PortfolioName string,"
     "IngestionTimestamp timestamp,"
     "BatchId string,"
     "DUnderLawKey string,"
     "CDI string,"
     "TaxRegistrationNumber string,"
     "ApplicationNumber string,"
     "ApplicationStatusCode string,"
     "ApplicationStatus string,"
     "ApplicationStatusDate date,"
     "TypeOfCourt string,"
     "CourtDate date,"
     "RecordDate date,"
     "SubmissionDate date,"
     "InitiationDate date,"
     "CourtOutcomeCode string,"
     "CourtOutcome string,"
     "CompletionReasonCode int,"
     "CompletionReason string,"
     "LawSystem string,"
     "PlatformName string,"
     "InLawFlag int,"
     "LawStatusCode string,"
     "LawStatus string,"
     "CalcLawStatusCode string,"
     "CalcLawStatus string,"
     "UnderLawType string,"
     "UnderLawTypeCode string,"
     "TemporaryCourtDecisionDate timestamp,"
     "DeductionDate timestamp,"
     "CountyCourtCode string,"
     "CountyCourt string,"
     "InterbankStatus3869Code string,"
     "InterbankStatus3869 string,"
     "SubmissionDateGroup string,"
     "ResubmissionStatusFlag string,"
     "ResubmissionStatusCode string,"
     "ResubmissionStage string,"
     "AppealCourtDate timestamp,"
     "CepalEligibilityDate timestamp,"
     "BusinessDate date"
    )
    

##### Create - Delete Delta Table

In [0]:
partition_columns = ["BusinessDate"]
derivative_table_path = ""
 
# ================== Create Delta Table ==================
if unity == "True":
    derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_unity_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
else:
    derivative_table_path = f"s3://{env}-ec1-cepal-derivative-zone/{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
 
 
# ================== Delete From Delta Table ==================
 
transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

##### Create wanted Delta Tables with selected fields

###### dunderlaw_qca

In [0]:
underlaw_qca=spark.sql(
   f"""SELECT                 
        UnderLawId,
        SourceSystem,
        PortfolioKey,
        PortfolioName,
        BusinessDate,
        UnderLawKey,
        CDI,
        TaxRegistrationNumber,
        ApplicationNumber,
        ApplicationStatusCode,
        ApplicationStatus,
        ApplicationStatusDate,
        TypeOfCourt,
        CourtDate,
        RecordDate,
        InitiationDate,
        CourtOutcomeCode,
        CourtOutcome,
        CompletionReasonCode,
        CompletionReason,
        LawSystem,
        PlatformName,
        InLawFLag,
        LawStatusCode,
        LawStatus,
        CalcLawStatusCode,
        CalcLawStatus,
        SubmissionDate,
        UnderLawType,
        UnderLawTypeCode,
        TemporaryCourtDecisionDate,
        DeductionDate,
        CountyCourtCode,
        CountyCourt,
        InterbankStatus3869Code,
        InterbankStatus3869,
        SubmissionDateGroup,
        ResubmissionStatusFlag,
        ResubmissionStatusCode,
        ResubmissionStage,
        AppealCourtDate
    FROM `{env}-cdh`.primary.underlaw
    WHERE BusinessDate = {run_date!r} AND SourceSystem="qca"
   """
)



#================== dunderlaw_qca =============================
underlaw_qca_primary_key_fields =[
    underlaw_qca.UnderLawType,
    underlaw_qca.CDI,
]

underlaw_qca=underlaw_qca.withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *underlaw_qca_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *underlaw_qca_primary_key_fields
    )
).withColumn("ApplicationStatusDate_calc",
             when(underlaw_qca.LawStatusCode.isin([21735, 21736]), underlaw_qca.SubmissionDate)
            .when(underlaw_qca.LawStatusCode.isin([22535, 21734]), underlaw_qca.CourtDate)
            .when(underlaw_qca.LawStatusCode.isin([21732, 21733, 22692, 22694]), underlaw_qca.SubmissionDate)
            .when(underlaw_qca.LawStatusCode == 22691, underlaw_qca.CourtDate)
            .when(underlaw_qca.LawStatusCode.isin([22695, 22696]), underlaw_qca.CourtDate)
            .when(underlaw_qca.LawStatusCode == 22693, when(underlaw_qca.DeductionDate.isNull(), underlaw_qca.CourtDate).otherwise             (underlaw_qca.DeductionDate))
            .when((underlaw_qca.LawStatusCode == 22534), (when(underlaw_qca.CourtDate.isNull(), underlaw_qca.SubmissionDate)).otherwise             (underlaw_qca.CourtDate)).otherwise(lit('1900-01-01'))
)



dunderlaw_qca= underlaw_qca.select(
    underlaw_qca.DUnderLawId.alias("DUnderLawId"),
    lit("qca").alias("SourceSystem"),
    underlaw_qca.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_qca.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_qca.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_qca.CDI.cast("string").alias("CDI"),
    underlaw_qca.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_qca.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_qca.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_qca.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_qca.ApplicationStatusDate_calc.cast("date").alias("ApplicationStatusDate"),
    underlaw_qca.TypeOfCourt.cast("string").alias("TypeOfCourt"),
    underlaw_qca.CourtDate.cast("date").alias("CourtDate"),
    underlaw_qca.RecordDate.cast("date").alias("RecordDate"),
    underlaw_qca.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_qca.InitiationDate.cast("date").alias("InitiationDate"),
    underlaw_qca.CourtOutcomeCode.cast("string").alias("CourtOutcomeCode"),
    underlaw_qca.CourtOutcome.cast("string").alias("CourtOutcome"),
    underlaw_qca.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_qca.CompletionReason.alias("CompletionReason"),
    underlaw_qca.LawSystem.alias("LawSystem"),
    underlaw_qca.PlatformName.alias("PlatformName"),
    underlaw_qca.InLawFLag.cast("int").alias("InLawFlag"),
    underlaw_qca.LawStatusCode.cast("string").alias("LawStatusCode"),
    underlaw_qca.LawStatus.cast("string").alias("LawStatus"),
    underlaw_qca.CalcLawStatusCode.alias("CalcLawStatusCode"),
    underlaw_qca.CalcLawStatus.alias("CalcLawStatus"),
    lit("N3869").cast("string").alias("UnderLawType"),
    lit("N3869").cast("string").alias("UnderLawTypeCode"),
    underlaw_qca.TemporaryCourtDecisionDate.alias("TemporaryCourtDecisionDate"),
    underlaw_qca.DeductionDate.alias("DeductionDate"),
    underlaw_qca.CountyCourtCode.alias("CountyCourtCode"),
    underlaw_qca.CountyCourt.alias("CountyCourt"),
    underlaw_qca.InterbankStatus3869Code.alias("InterbankStatus3869Code"),
    underlaw_qca.InterbankStatus3869.alias("InterbankStatus3869"),
    underlaw_qca.SubmissionDateGroup.alias("SubmissionDateGroup"),
    underlaw_qca.ResubmissionStatusFlag.alias("ResubmissionStatusFlag"),
    underlaw_qca.ResubmissionStatusCode.alias("ResubmissionStatusCode"),
    underlaw_qca.ResubmissionStage.alias("ResubmissionStage"),
    underlaw_qca.AppealCourtDate.alias("AppealCourtDate"),
    lit(None).alias("CepalEligibilityDate"),
    underlaw_qca.BusinessDate.alias("BusinessDate")
)
underlaw_qca=underlaw_qca.distinct()

###### dunderlaw_process

In [0]:
underlaw_process=spark.sql(
   f"""SELECT                 
        u.UnderLawId,
        u.SourceSystem,
        u.PortfolioKey,
        u.PortfolioName,
        u.BusinessDate,
        u.UnderLawKey,
        u.CDI,
        u.TaxRegistrationNumber,
        u.ApplicationNumber,
        u.ApplicationStatusCode,
        u.ApplicationStatus,
        u.ApplicationStatusDate,
        u.TypeOfCourt,
        u.CourtDate,
        u.RecordDate,
        u.InitiationDate,
        u.CourtOutcomeCode,
        u.CourtOutcome,
        u.CompletionReasonCode,
        u.CompletionReason,
        u.LawSystem,
        u.PlatformName,
        u.InLawFLag,
        u.LawStatusCode,
        u.LawStatus,
        u.CalcLawStatusCode,
        u.CalcLawStatus,
        u.SubmissionDate,
        u.UnderLawType,
        u.UnderLawTypeCode,
        u.TemporaryCourtDecisionDate,
        u.DeductionDate,
        u.CountyCourtCode,
        u.CountyCourt,
        u.InterbankStatus3869Code,
        u.InterbankStatus3869,
        u.SubmissionDateGroup,
        u.ResubmissionStatusFlag,
        u.ResubmissionStatusCode,
        u.ResubmissionStage,
        u.AppealCourtDate
    FROM `{env}-cdh`.primary.underlaw u
    INNER JOIN `{env}-cdh`.primary.party p ON p.CDI=u.CDI
    WHERE u.BusinessDate = {run_date!r} 
    AND u.SourceSystem="process"
    AND (coalesce(u.TaxRegistrationNumber, '') = '' OR u.TaxRegistrationNumber = p.TaxRegistrationNumber)

   """
)


#================== dunderlaw_process =============================
underlaw_process_primary_key_fields =[
    underlaw_process.UnderLawType,
    underlaw_process.CDI,
]

underlaw_process=underlaw_process.withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *underlaw_process_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *underlaw_process_primary_key_fields
    )
)


dunderlaw_process= underlaw_process.select(
    underlaw_process.DUnderLawId.alias("DUnderLawId"),
    lit("process").alias("SourceSystem"),
    underlaw_process.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_process.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_process.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_process.CDI.cast("string").alias("CDI"),
    underlaw_process.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_process.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_process.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_process.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_process.ApplicationStatusDate.cast("date").alias("ApplicationStatusDate"),
    underlaw_process.TypeOfCourt.cast("string").alias("TypeOfCourt"),
    underlaw_process.CourtDate.cast("date").alias("CourtDate"),
    underlaw_process.RecordDate.cast("date").alias("RecordDate"),
    underlaw_process.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_process.InitiationDate.cast("date").alias("InitiationDate"),
    underlaw_process.CourtOutcomeCode.cast("string").alias("CourtOutcomeCode"),
    underlaw_process.CourtOutcome.cast("string").alias("CourtOutcome"),
    underlaw_process.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_process.CompletionReason.alias("CompletionReason"),
    underlaw_process.LawSystem.alias("LawSystem"),
    underlaw_process.PlatformName.alias("PlatformName"),
    underlaw_process.InLawFLag.cast("int").alias("InLawFlag"),
    underlaw_process.LawStatusCode.cast("string").alias("LawStatusCode"),
    underlaw_process.LawStatus.cast("string").alias("LawStatus"),
    underlaw_process.CalcLawStatusCode.alias("CalcLawStatusCode"),
    underlaw_process.CalcLawStatus.alias("CalcLawStatus"),
    lit("N3869").cast("string").alias("UnderLawType"),
    lit("N3869").cast("string").alias("UnderLawTypeCode"),
    underlaw_process.TemporaryCourtDecisionDate.alias("TemporaryCourtDecisionDate"),
    underlaw_process.DeductionDate.alias("DeductionDate"),
    underlaw_process.CountyCourtCode.alias("CountyCourtCode"),
    underlaw_process.CountyCourt.alias("CountyCourt"),
    underlaw_process.InterbankStatus3869Code.alias("InterbankStatus3869Code"),
    underlaw_process.InterbankStatus3869.alias("InterbankStatus3869"),
    underlaw_process.SubmissionDateGroup.alias("SubmissionDateGroup"),
    underlaw_process.ResubmissionStatusFlag.alias("ResubmissionStatusFlag"),
    underlaw_process.ResubmissionStatusCode.alias("ResubmissionStatusCode"),
    underlaw_process.ResubmissionStage.alias("ResubmissionStage"),
    underlaw_process.AppealCourtDate.alias("AppealCourtDate"),
    lit(None).alias("CepalEligibilityDate"),
    underlaw_process.BusinessDate.alias("BusinessDate")
).distinct()


###### dunderlaw_rpt

In [0]:
party=spark.sql(
   f"""SELECT  
        CDI,
        TaxRegistrationNumber,
        PortfolioName,
        PortfolioKey           
    FROM `{env}-cdh`.primary.party
    WHERE BusinessDate = {run_date!r} 
    AND CDI IS NOT NULL
    AND PortfolioName not in ("Core Retail","Core Wholesale")
    AND SourceSystem = 'profits'
   """
)
party=party.withColumnRenamed("CDI","CDI_party").withColumnRenamed("TaxRegistrationNumber","TaxRegistrationNumber_party")

# ================== underlaw_rpt ==================
underlaw_rpt=spark.sql(
   f"""SELECT                 
    UnderLawId,
    CDI,
    UnderLawKey,
    SourceSystem,
    TaxRegistrationNumber,
    ApplicationNumber,
    UnderLawType,
    UnderLawTypeCode,
    ApplicationStatusCode,
    ApplicationStatus,
    ApplicationStatusDate,
    SubmissionDate,
    InitiationDate,
    CompletionReasonCode,
    CompletionReason,
    LawSystem,
    PlatformName,
    CepalEligibilityDate,
    BusinessDate
    FROM `{env}-cdh`.primary.underlaw
    WHERE BusinessDate = {run_date!r} AND SourceSystem="rpt"
   """
)


# ================ N4734/2020 | N4790/2021 ===========

underlaw_n4734_n4790 = underlaw_rpt.filter((underlaw_rpt.UnderLawType.isin("N4734/2020","N4790/2021"))
         & (underlaw_rpt.ApplicationStatusCode.isin(7, 99)))


underlaw_n4734_n4790 = underlaw_n4734_n4790.filter(underlaw_n4734_n4790.CepalEligibilityDate.isNotNull())
underlaw_n4734_n4790 = underlaw_n4734_n4790.filter(underlaw_n4734_n4790.CDI.isNotNull())

window_spec_n4734_n4790 = Window.partitionBy(underlaw_n4734_n4790.CDI).orderBy(desc(underlaw_n4734_n4790.CepalEligibilityDate))

underlaw_n4734_n4790=underlaw_n4734_n4790.withColumn(
    "rn",row_number().over(window_spec_n4734_n4790)
)

underlaw_n4734_n4790=underlaw_n4734_n4790.filter(col("rn")==1)

underlaw_n4734_n4790 =underlaw_n4734_n4790.join(
    party,
    party.CDI_party == underlaw_n4734_n4790.CDI,
    "left",
)

n4734_n4790_primary_key_fields =[
    underlaw_n4734_n4790.UnderLawType,
    underlaw_n4734_n4790.CDI,
]


underlaw_n4734_n4790=underlaw_n4734_n4790.withColumn(
    "InLawFlag",
    when(underlaw_n4734_n4790.ApplicationStatusCode == "7",1)
    .otherwise(0)
).withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *n4734_n4790_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *n4734_n4790_primary_key_fields
    )
)




underlaw_n4734_n4790_df= underlaw_n4734_n4790.select(
    underlaw_n4734_n4790.DUnderLawId.alias("DUnderLawId"),
    lit("rpt").alias("SourceSystem"),
    underlaw_n4734_n4790.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_n4734_n4790.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_n4734_n4790.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_n4734_n4790.CDI.cast("string").alias("CDI"),
    underlaw_n4734_n4790.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_n4734_n4790.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_n4734_n4790.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_n4734_n4790.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_n4734_n4790.ApplicationStatusDate.cast("date").alias("ApplicationStatusDate"),
    lit(None).cast("string").alias("TypeOfCourt"),
    lit(None).cast("date").alias("CourtDate"),
    lit(None).cast("date").alias("RecordDate"),
    underlaw_n4734_n4790.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_n4734_n4790.InitiationDate.cast("date").alias("InitiationDate"),
    lit(None).cast("string").alias("CourtOutcomeCode"),
    lit(None).cast("string").alias("CourtOutcome"),
    underlaw_n4734_n4790.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_n4734_n4790.CompletionReason.alias("CompletionReason"),
    underlaw_n4734_n4790.LawSystem.alias("LawSystem"),
    underlaw_n4734_n4790.PlatformName.alias("PlatformName"),
    underlaw_n4734_n4790.InLawFlag.cast("int").alias("InLawFlag"),
    lit(None).cast("string").alias("LawStatusCode"),
    lit(None).cast("string").alias("LawStatus"),
    lit(None).alias("CalcLawStatusCode"),
    lit(None).alias("CalcLawStatus"),
    underlaw_n4734_n4790.UnderLawType.cast("string").alias("UnderLawType"),
    underlaw_n4734_n4790.UnderLawTypeCode.cast("string").alias("UnderLawTypeCode"),
    lit(None).alias("TemporaryCourtDecisionDate"),
    lit(None).alias("DeductionDate"),
    lit(None).alias("CountyCourtCode"),
    lit(None).alias("CountyCourt"),
    lit(None).alias("InterbankStatus3869Code"),
    lit(None).alias("InterbankStatus3869"),
    lit(None).alias("SubmissionDateGroup"),
    lit(None).alias("ResubmissionStatusFlag"),
    lit(None).alias("ResubmissionStatusCode"),
    lit(None).alias("ResubmissionStage"),
    lit(None).alias("AppealCourtDate"),
    underlaw_n4734_n4790.CepalEligibilityDate.alias("CepalEligibilityDate"),
    underlaw_n4734_n4790.BusinessDate.alias("BusinessDate")
)

underlaw_n4734_n4790_df=underlaw_n4734_n4790_df.distinct()

#================== Ν4605/2019 ==============================

underlaw_n4605=underlaw_rpt.filter(underlaw_rpt.UnderLawType == "N4605/2019")
underlaw_n4605 = underlaw_n4605.filter(underlaw_n4605.CDI.isNotNull())

window_spec_n4605 = Window.partitionBy(underlaw_n4605.CDI).orderBy(desc(underlaw_n4605.ApplicationStatusDate),desc(coalesce(underlaw_n4605.SubmissionDate,lit('1900-01-01'))),desc(underlaw_n4605.InitiationDate))

underlaw_n4605=underlaw_n4605.withColumn(
    "rn",row_number().over(window_spec_n4605)
)

underlaw_n4605=underlaw_n4605.filter(col("rn")==1)

underlaw_n4605 =underlaw_n4605.join(
    party,
    party.CDI_party == underlaw_n4605.CDI,
    "left",
)
n4605_primary_key_fields =[
    underlaw_n4605.UnderLawType,
    underlaw_n4605.CDI,
]

underlaw_n4605=underlaw_n4605.withColumn(
    "InLawFlag",
    when(underlaw_n4605.ApplicationStatus.isin(
            "Έγκριση Χορήγησης Συνεισφοράς Δημοσίου",
            "Καταχώρηση Δικαστικής Απόφασης",
            "Παρακολούθηση Συνεισφοράς Δημοσίου"),1)
    .otherwise(0)
).withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *n4605_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *n4605_primary_key_fields
    )
)



underlaw_n4605_df= underlaw_n4605.select(
    underlaw_n4605.DUnderLawId.alias("DUnderLawId"),
    lit("rpt").alias("SourceSystem"),
    underlaw_n4605.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_n4605.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_n4605.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_n4605.CDI.cast("string").alias("CDI"),
    underlaw_n4605.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_n4605.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_n4605.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_n4605.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_n4605.ApplicationStatusDate.cast("date").alias("ApplicationStatusDate"),
    lit(None).cast("string").alias("TypeOfCourt"),
    lit(None).cast("date").alias("CourtDate"),
    lit(None).cast("date").alias("RecordDate"),
    underlaw_n4605.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_n4605.InitiationDate.cast("date").alias("InitiationDate"),
    lit(None).cast("string").alias("CourtOutcomeCode"),
    lit(None).cast("string").alias("CourtOutcome"),
    underlaw_n4605.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_n4605.CompletionReason.alias("CompletionReason"),
    underlaw_n4605.LawSystem.alias("LawSystem"),
    underlaw_n4605.PlatformName.alias("PlatformName"),
    underlaw_n4605.InLawFlag.cast("int").alias("InLawFlag"),
    lit(None).cast("string").alias("LawStatusCode"),
    lit(None).cast("string").alias("LawStatus"),
    lit(None).alias("CalcLawStatusCode"),
    lit(None).alias("CalcLawStatus"),
    underlaw_n4605.UnderLawType.cast("string").alias("UnderLawType"),
    underlaw_n4605.UnderLawTypeCode.cast("string").alias("UnderLawTypeCode"),
    lit(None).alias("TemporaryCourtDecisionDate"),
    lit(None).alias("DeductionDate"),
    lit(None).alias("CountyCourtCode"),
    lit(None).alias("CountyCourt"),
    lit(None).alias("InterbankStatus3869Code"),
    lit(None).alias("InterbankStatus3869"),
    lit(None).alias("SubmissionDateGroup"),
    lit(None).alias("ResubmissionStatusFlag"),
    lit(None).alias("ResubmissionStatusCode"),
    lit(None).alias("ResubmissionStage"),
    lit(None).alias("AppealCourtDate"),
    underlaw_n4605.CepalEligibilityDate.alias("CepalEligibilityDate"),
    underlaw_n4605.BusinessDate.alias("BusinessDate")
)

underlaw_n4605_df=underlaw_n4605_df.distinct()

In [0]:
# ================== underlaw_cepalbiplatform =================
underlaw_cepalbiplatform=spark.sql(
   f"""SELECT                 
    UnderLawId,
    CDI,
    UnderLawKey,
    SourceSystem,
    TaxRegistrationNumber,
    ApplicationNumber,
    UnderLawType,
    UnderLawTypeCode,
    ApplicationStatusCode,
    ApplicationStatus,
    ApplicationStatusDate,
    SubmissionDate,
    InitiationDate,
    CompletionReasonCode,
    CompletionReason,
    LawSystem,
    PlatformName,
    CepalEligibilityDate,
    BusinessDate
    FROM `{env}-cdh`.primary.underlaw
    WHERE BusinessDate = {run_date!r} AND SourceSystem="cepalbiplatform"
   """
)





#=================== N4738/2020-Πτώχευση  ==========================

underlaw_n4738_first=underlaw_cepalbiplatform.filter((underlaw_cepalbiplatform.UnderLawType == "N4738/2020")
    & (underlaw_cepalbiplatform.LawSystem=="Πτώχευση")
    & (underlaw_cepalbiplatform.ApplicationStatusCode=="3"))

window_spec_n4738_first = Window.partitionBy(underlaw_n4738_first.TaxRegistrationNumber).orderBy(desc(coalesce(underlaw_n4738_first.SubmissionDate,lit('1900-01-01'))))

underlaw_n4738_first=underlaw_n4738_first.withColumn(
    "rn",row_number().over(window_spec_n4738_first)
)

underlaw_n4738_first=underlaw_n4738_first.filter(col("rn")==1)

underlaw_n4738_first=underlaw_n4738_first.join(
    party,
    party.TaxRegistrationNumber_party == underlaw_n4738_first.TaxRegistrationNumber,
    "inner",
)

n4738_first_primary_key_fields =[
    underlaw_n4738_first.UnderLawType,
    underlaw_n4738_first.LawSystem,
    underlaw_n4738_first.CDI_party
]

underlaw_n4738_first=underlaw_n4738_first.withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *n4738_first_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *n4738_first_primary_key_fields
    )
)

underlaw_n4738_first_df= underlaw_n4738_first.select(
    underlaw_n4738_first.DUnderLawId.alias("DUnderLawId"),
    lit("cepalbiplatform").alias("SourceSystem"),
    underlaw_n4738_first.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_n4738_first.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_n4738_first.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_n4738_first.CDI_party.cast("string").alias("CDI"),
    underlaw_n4738_first.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_n4738_first.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_n4738_first.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_n4738_first.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_n4738_first.ApplicationStatusDate.cast("date").alias("ApplicationStatusDate"),
    lit(None).cast("string").alias("TypeOfCourt"),
    lit(None).cast("date").alias("CourtDate"),
    lit(None).cast("date").alias("RecordDate"),
    underlaw_n4738_first.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_n4738_first.InitiationDate.cast("date").alias("InitiationDate"),
    lit(None).cast("string").alias("CourtOutcomeCode"),
    lit(None).cast("string").alias("CourtOutcome"),
    underlaw_n4738_first.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_n4738_first.CompletionReason.alias("CompletionReason"),
    underlaw_n4738_first.LawSystem.alias("LawSystem"),
    underlaw_n4738_first.PlatformName.alias("PlatformName"),
    lit("1").cast("int").alias("InLawFlag"),
    lit(None).cast("string").alias("LawStatusCode"),
    lit(None).cast("string").alias("LawStatus"),
    lit(None).alias("CalcLawStatusCode"),
    lit(None).alias("CalcLawStatus"),
    underlaw_n4738_first.UnderLawType.cast("string").alias("UnderLawType"),
    underlaw_n4738_first.UnderLawTypeCode.cast("string").alias("UnderLawTypeCode"),
    lit(None).alias("TemporaryCourtDecisionDate"),
    lit(None).alias("DeductionDate"),
    lit(None).alias("CountyCourtCode"),
    lit(None).alias("CountyCourt"),
    lit(None).alias("InterbankStatus3869Code"),
    lit(None).alias("InterbankStatus3869"),
    lit(None).alias("SubmissionDateGroup"),
    lit(None).alias("ResubmissionStatusFlag"),
    lit(None).alias("ResubmissionStatusCode"),
    lit(None).alias("ResubmissionStage"),
    lit(None).alias("AppealCourtDate"),
    underlaw_n4738_first.CepalEligibilityDate.alias("CepalEligibilityDate"),
    underlaw_n4738_first.BusinessDate.alias("BusinessDate")
)


underlaw_n4738_first_df=underlaw_n4738_first_df.distinct()

#===================== N4738/2020 - Εξυγίανση ======================

underlaw_n4738_second=underlaw_cepalbiplatform.filter((underlaw_cepalbiplatform.UnderLawType == "N4738/2020")
    & (underlaw_cepalbiplatform.LawSystem=="Εξυγίανση")
    & (underlaw_cepalbiplatform.ApplicationStatusCode=="2"))

window_spec_n4738_second = Window.partitionBy(underlaw_n4738_second.TaxRegistrationNumber).orderBy(desc(coalesce(underlaw_n4738_second.SubmissionDate,lit('1900-01-01'))))

underlaw_n4738_second=underlaw_n4738_second.withColumn(
    "rn",row_number().over(window_spec_n4738_second)
)



underlaw_n4738_second=underlaw_n4738_second.filter(col("rn")==1)

underlaw_n4738_second=underlaw_n4738_second.join(
    party,
    party.TaxRegistrationNumber_party == underlaw_n4738_second.TaxRegistrationNumber,
    "inner",
)

n4738_second_primary_key_fields =[
    underlaw_n4738_second.UnderLawType,
    underlaw_n4738_second.LawSystem,
    underlaw_n4738_second.CDI_party
]

underlaw_n4738_second=underlaw_n4738_second.withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *n4738_second_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *n4738_second_primary_key_fields
    )
)


underlaw_n4738_second_df= underlaw_n4738_second.select(
    underlaw_n4738_second.DUnderLawId.alias("DUnderLawId"),
    lit("cepalbiplatform").alias("SourceSystem"),
    underlaw_n4738_second.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_n4738_second.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_n4738_second.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_n4738_second.CDI_party.cast("string").alias("CDI"),
    underlaw_n4738_second.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_n4738_second.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_n4738_second.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_n4738_second.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_n4738_second.ApplicationStatusDate.cast("date").alias("ApplicationStatusDate"),
    lit(None).cast("string").alias("TypeOfCourt"),
    lit(None).cast("date").alias("CourtDate"),
    lit(None).cast("date").alias("RecordDate"),
    underlaw_n4738_second.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_n4738_second.InitiationDate.cast("date").alias("InitiationDate"),
    lit(None).cast("string").alias("CourtOutcomeCode"),
    lit(None).cast("string").alias("CourtOutcome"),
    underlaw_n4738_second.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_n4738_second.CompletionReason.alias("CompletionReason"),
    underlaw_n4738_second.LawSystem.alias("LawSystem"),
    underlaw_n4738_second.PlatformName.alias("PlatformName"),
    lit("1").cast("int").alias("InLawFlag"),
    lit(None).cast("string").alias("LawStatusCode"),
    lit(None).cast("string").alias("LawStatus"),
    lit(None).alias("CalcLawStatusCode"),
    lit(None).alias("CalcLawStatus"),
    underlaw_n4738_second.UnderLawType.cast("string").alias("UnderLawType"),
    underlaw_n4738_second.UnderLawType.cast("string").alias("UnderLawTypeCode"),
    lit(None).alias("TemporaryCourtDecisionDate"),
    lit(None).alias("DeductionDate"),
    lit(None).alias("CountyCourtCode"),
    lit(None).alias("CountyCourt"),
    lit(None).alias("InterbankStatus3869Code"),
    lit(None).alias("InterbankStatus3869"),
    lit(None).alias("SubmissionDateGroup"),
    lit(None).alias("ResubmissionStatusFlag"),
    lit(None).alias("ResubmissionStatusCode"),
    lit(None).alias("ResubmissionStage"),
    lit(None).alias("AppealCourtDate"),
    underlaw_n4738_second.CepalEligibilityDate.alias("CepalEligibilityDate"),
    underlaw_n4738_second.BusinessDate.alias("BusinessDate")
)

underlaw_n4738_second_df=underlaw_n4738_second_df.distinct()

In [0]:
#======================= N4738/2020 - Εξωδικαστικός ===============================================

underlaw_n4738_third = underlaw_cepalbiplatform.filter(
(underlaw_cepalbiplatform.UnderLawType == "N4738/2020") &
(underlaw_cepalbiplatform.LawSystem == "Εξωδικαστικός μηχανισμός") &
((underlaw_cepalbiplatform.ApplicationStatusCode < 98) | 
(underlaw_cepalbiplatform.ApplicationStatusCode == 99))
)



window_spec_n4738_third = Window.partitionBy(underlaw_n4738_third.TaxRegistrationNumber).orderBy(desc(underlaw_n4738_third.ApplicationStatusDate),desc(coalesce(underlaw_n4738_third.SubmissionDate,lit('1900-01-01'))),desc(underlaw_n4738_third.InitiationDate))

underlaw_n4738_third=underlaw_n4738_third.withColumn(
    "rn",row_number().over(window_spec_n4738_third)
)

underlaw_n4738_third=underlaw_n4738_third.filter(col("rn")==1)

underlaw_n4738_third=underlaw_n4738_third.join(
    party,
    party.TaxRegistrationNumber_party == underlaw_n4738_third.TaxRegistrationNumber,
    "inner",
)


n4738_third_primary_key_fields =[
    underlaw_n4738_third.UnderLawType,
    underlaw_n4738_third.LawSystem,
    underlaw_n4738_third.CDI_party
]

underlaw_n4738_third=underlaw_n4738_third.withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *n4738_third_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *n4738_third_primary_key_fields
    )
)

underlaw_n4738_third_df= underlaw_n4738_third.select(
    underlaw_n4738_third.DUnderLawId.alias("DUnderLawId"),
    lit("cepalbiplatform").alias("SourceSystem"),
    underlaw_n4738_third.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_n4738_third.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_n4738_third.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_n4738_third.CDI_party.cast("string").alias("CDI"),
    underlaw_n4738_third.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_n4738_third.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_n4738_third.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_n4738_third.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_n4738_third.ApplicationStatusDate.cast("date").alias("ApplicationStatusDate"),
    lit(None).cast("string").alias("TypeOfCourt"),
    lit(None).cast("date").alias("CourtDate"),
    lit(None).cast("date").alias("RecordDate"),
    underlaw_n4738_third.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_n4738_third.InitiationDate.cast("date").alias("InitiationDate"),
    lit(None).cast("string").alias("CourtOutcomeCode"),
    lit(None).cast("string").alias("CourtOutcome"),
    underlaw_n4738_third.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_n4738_third.CompletionReason.alias("CompletionReason"),
    underlaw_n4738_third.LawSystem.alias("LawSystem"),
    underlaw_n4738_third.PlatformName.alias("PlatformName"),
    lit("1").cast("int").alias("InLawFlag"),
    lit(None).cast("string").alias("LawStatusCode"),
    lit(None).cast("string").alias("LawStatus"),
    lit(None).alias("CalcLawStatusCode"),
    lit(None).alias("CalcLawStatus"),
    underlaw_n4738_third.UnderLawType.cast("string").alias("UnderLawType"),
    underlaw_n4738_third.UnderLawType.cast("string").alias("UnderLawTypeCode"),
    lit(None).alias("TemporaryCourtDecisionDate"),
    lit(None).alias("DeductionDate"),
    lit(None).alias("CountyCourtCode"),
    lit(None).alias("CountyCourt"),
    lit(None).alias("InterbankStatus3869Code"),
    lit(None).alias("InterbankStatus3869"),
    lit(None).alias("SubmissionDateGroup"),
    lit(None).alias("ResubmissionStatusFlag"),
    lit(None).alias("ResubmissionStatusCode"),
    lit(None).alias("ResubmissionStage"),
    lit(None).alias("AppealCourtDate"),
    underlaw_n4738_third.CepalEligibilityDate.alias("CepalEligibilityDate"),
    underlaw_n4738_third.BusinessDate.alias("BusinessDate")
)

underlaw_n4738_third_df=underlaw_n4738_third_df.distinct()

#================== N4738/2020 - Βεβαίωση Ευαλώτου =============================

underlaw_n4738_fourth=underlaw_cepalbiplatform.filter((underlaw_cepalbiplatform.UnderLawType == "N4738/2020")
    & (underlaw_cepalbiplatform.LawSystem=="Βεβαίωση ευάλωτου")
    & (underlaw_cepalbiplatform.ApplicationStatusCode=="99")
    & (underlaw_cepalbiplatform.CompletionReasonCode =="502")
    )

window_spec_n4738_fourth = Window.partitionBy(underlaw_n4738_fourth.TaxRegistrationNumber).orderBy(desc(coalesce(underlaw_n4738_fourth.SubmissionDate,lit('1900-01-01'))))

underlaw_n4738_fourth=underlaw_n4738_fourth.withColumn(
    "rn",row_number().over(window_spec_n4738_fourth)
)

underlaw_n4738_fourth=underlaw_n4738_fourth.filter(col("rn")==1)

underlaw_n4738_fourth=underlaw_n4738_fourth.join(
    party,
    party.TaxRegistrationNumber_party == underlaw_n4738_fourth.TaxRegistrationNumber,
    "inner",
)




n4738_fourth_primary_key_fields =[
    underlaw_n4738_fourth.UnderLawType,
    underlaw_n4738_fourth.LawSystem,
    underlaw_n4738_fourth.CDI_party
]

underlaw_n4738_fourth=underlaw_n4738_fourth.withColumn(
    "DUnderLawId",
    sha2(
        concat_ws(
            ";", *n4738_fourth_primary_key_fields
        ),
        256,
    ),
).withColumn(
    "DUnderLawKey",
    concat_ws(
        ";", *n4738_fourth_primary_key_fields
    )
)

underlaw_n4738_fourth_df= underlaw_n4738_fourth.select(
    underlaw_n4738_fourth.DUnderLawId.alias("DUnderLawId"),
    lit("cepalbiplatform").alias("SourceSystem"),
    underlaw_n4738_fourth.PortfolioKey.cast("string").alias("PortfolioKey"),
    underlaw_n4738_fourth.PortfolioName.cast("string").alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    underlaw_n4738_fourth.DUnderLawKey.alias("DUnderLawKey"),
    underlaw_n4738_fourth.CDI_party.cast("string").alias("CDI"),
    underlaw_n4738_fourth.TaxRegistrationNumber.alias("TaxRegistrationNumber"),
    underlaw_n4738_fourth.ApplicationNumber.alias("ApplicationNumber"),
    underlaw_n4738_fourth.ApplicationStatusCode.cast("string").alias("ApplicationStatusCode"),
    underlaw_n4738_fourth.ApplicationStatus.cast("string").alias("ApplicationStatus"),
    underlaw_n4738_fourth.ApplicationStatusDate.cast("date").alias("ApplicationStatusDate"),
    lit(None).cast("string").alias("TypeOfCourt"),
    lit(None).cast("date").alias("CourtDate"),
    lit(None).cast("date").alias("RecordDate"),
    underlaw_n4738_fourth.SubmissionDate.cast("date").alias("SubmissionDate"),
    underlaw_n4738_fourth.InitiationDate.cast("date").alias("InitiationDate"),
    lit(None).cast("string").alias("CourtOutcomeCode"),
    lit(None).cast("string").alias("CourtOutcome"),
    underlaw_n4738_fourth.CompletionReasonCode.alias("CompletionReasonCode"),
    underlaw_n4738_fourth.CompletionReason.alias("CompletionReason"),
    underlaw_n4738_fourth.LawSystem.alias("LawSystem"),
    underlaw_n4738_fourth.PlatformName.alias("PlatformName"),
    lit("0").cast("int").alias("InLawFlag"),
    lit(None).cast("string").alias("LawStatusCode"),
    lit(None).cast("string").alias("LawStatus"),
    lit(None).alias("CalcLawStatusCode"),
    lit(None).alias("CalcLawStatus"),
    underlaw_n4738_fourth.UnderLawType.cast("string").alias("UnderLawType"),
    underlaw_n4738_fourth.UnderLawType.cast("string").alias("UnderLawTypeCode"),
    lit(None).alias("TemporaryCourtDecisionDate"),
    lit(None).alias("DeductionDate"),
    lit(None).alias("CountyCourtCode"),
    lit(None).alias("CountyCourt"),
    lit(None).alias("InterbankStatus3869Code"),
    lit(None).alias("InterbankStatus3869"),
    lit(None).alias("SubmissionDateGroup"),
    lit(None).alias("ResubmissionStatusFlag"),
    lit(None).alias("ResubmissionStatusCode"),
    lit(None).alias("ResubmissionStage"),
    lit(None).alias("AppealCourtDate"),
    underlaw_n4738_fourth.CepalEligibilityDate.alias("CepalEligibilityDate"),
    underlaw_n4738_fourth.BusinessDate.alias("BusinessDate")
)

underlaw_n4738_fourth_df=underlaw_n4738_fourth_df.distinct()

# ========================== Append ==========================

dunderlaw_rpt_cepalbiplatform=underlaw_n4605_df.union(underlaw_n4734_n4790_df).union(underlaw_n4738_first_df
            ).union(underlaw_n4738_second_df).union(underlaw_n4738_third_df
            ).union(underlaw_n4738_fourth_df)



##### Final DataFrame dunderlaw

In [0]:
dunderlaw=dunderlaw_qca.union(dunderlaw_process).union(dunderlaw_rpt_cepalbiplatform)

# ------------------------------------------------------------------------------------
# De-duplicate the qca/process collision (Option 2: first non-null per column).
#
# Only 'qca' and 'process' can produce the SAME DUnderLawId (= hash of UnderLawType + CDI),
# because both emit UnderLawType = 'N3869' for the same CDI. 'cepalbiplatform' and 'rpt'
# always carry a different UnderLawType, so they never collide and must stay untouched.
#
# For each DUnderLawId we collapse the group to ONE row and, per column, take the first
# non-null value following the source priority qca > process. Single-source groups (only
# qca, only process, or any rpt/cepalbiplatform row) are returned exactly as-is, since the
# coalesce over a 1-row group just yields that row's own values.
# ------------------------------------------------------------------------------------

# Lower rank = higher priority. qca first (preserves the previous qca-preferred behaviour),
# then process. Any other source system is left after them but in practice never shares a
# DUnderLawId with these two.
_src_rank = (
    when(dunderlaw.SourceSystem == "qca", lit(0))
    .when(dunderlaw.SourceSystem == "process", lit(1))
    .otherwise(lit(2))
)
dunderlaw = dunderlaw.withColumn("_src_rank", _src_rank)

# Key columns are identical across the colliding rows -> just carry them through.
_key_cols = ["DUnderLawId", "DUnderLawKey", "CDI", "UnderLawType"]
# Per-column coalesce ordered by source priority for every non-key column.
_coalesce_cols = [c for c in dunderlaw.columns if c not in _key_cols + ["_src_rank"]]

_asc_window = Window.partitionBy("DUnderLawId").orderBy(col("_src_rank").asc())

_agg_exprs = []
for _c in _coalesce_cols:
    # first() with ignorenulls=True over the priority-ordered window = first non-null value
    # following qca > process. 'SourceSystem' therefore reflects the source that supplied
    # the winning (base) row.
    _agg_exprs.append(
        spark_first(col(_c), ignorenulls=True).over(_asc_window).alias(_c)
    )

_dedup_rank_window = Window.partitionBy("DUnderLawId").orderBy(col("_src_rank").asc())

dunderlaw = (
    dunderlaw.select(
        *_key_cols,
        *_agg_exprs,
        row_number().over(_dedup_rank_window).alias("_dedup_rn"),
    )
    .filter(col("_dedup_rn") == 1)
    .drop("_dedup_rn")
)

# Restore the original column order.
dunderlaw = dunderlaw.select(
    "DUnderLawId",
    "SourceSystem",
    "PortfolioKey",
    "PortfolioName",
    "IngestionTimestamp",
    "BatchId",
    "DUnderLawKey",
    "CDI",
    "TaxRegistrationNumber",
    "ApplicationNumber",
    "ApplicationStatusCode",
    "ApplicationStatus",
    "ApplicationStatusDate",
    "TypeOfCourt",
    "CourtDate",
    "RecordDate",
    "SubmissionDate",
    "InitiationDate",
    "CourtOutcomeCode",
    "CourtOutcome",
    "CompletionReasonCode",
    "CompletionReason",
    "LawSystem",
    "PlatformName",
    "InLawFlag",
    "LawStatusCode",
    "LawStatus",
    "CalcLawStatusCode",
    "CalcLawStatus",
    "UnderLawType",
    "UnderLawTypeCode",
    "TemporaryCourtDecisionDate",
    "DeductionDate",
    "CountyCourtCode",
    "CountyCourt",
    "InterbankStatus3869Code",
    "InterbankStatus3869",
    "SubmissionDateGroup",
    "ResubmissionStatusFlag",
    "ResubmissionStatusCode",
    "ResubmissionStage",
    "AppealCourtDate",
    "CepalEligibilityDate",
    "BusinessDate",
)

##### Write DataFrame to Delta Lake based on 'unity' value

In [0]:
if unity == "True":
    dunderlaw.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(derivative_table_path) # in order to write in delta_lake
else:
    dunderlaw.write \
        .format("delta") \
        .mode("append") \
        .save(derivative_table_path) # in order to write in delta_lake